# S-pre

Response diagrams, target quantiles and class labels.


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# six shear wall models (8-30 storeys) and six wall-frame models (8-25 storeys)
TRAIN_SET = ['W1','W2','W4','W6','WF1','WF2','WF4','WF6']
VALID_SET = ['W3','W5','WF3','WF5']

RAW_DIR    = 'data/gm/'
RDTD_DIR   = 'data/rdtd/'
TARGET_DIR = 'data/target/'

CFG = dict(f_min=0.1, f_max=10.0, n_freq=490, dt=0.01, duration=40.0,
           zeta=0.05, sg_win=51, sg_poly=3, levels=50, dpi=300, size=(8, 3))

for d in (RDTD_DIR, TARGET_DIR):
    os.makedirs(d, exist_ok=True)


In [ ]:
def sdof(omega, dt, acc, zeta=CFG['zeta']):
    """Displacement of a damped SDOF oscillator under base acceleration."""
    u = np.zeros_like(acc)
    for j in range(1, len(acc) - 1):
        u[j+1] = (2*u[j] - u[j-1] + dt**2 * (-acc[j] - omega**2 * u[j])) \
                 / (1.0 + zeta * omega * dt)
    return u

def response_diagram(path, cfg=CFG):
    """Elastic response diagram in the time domain.

    The peak ground acceleration is aligned to the horizontal centre of a
    ``duration`` window; the vertical axis spans ``f_min``-``f_max`` Hz on a
    logarithmic scale, and each cell holds the SDOF displacement normalised by
    its peak.
    """
    acc = savgol_filter(np.loadtxt(path, skiprows=1), cfg['sg_win'], cfg['sg_poly'])
    half = int(cfg['duration'] / 2 / cfg['dt'])
    peak = int(np.argmax(np.abs(acc)))
    acc = np.pad(acc, (max(0, half - peak), max(0, peak + half - len(acc))), 'constant')
    acc = acc[max(0, peak - half):peak + half]

    freqs = np.linspace(cfg['f_min'], cfg['f_max'], cfg['n_freq'])
    z = np.zeros((len(freqs), int(cfg['duration'] / cfg['dt'])))
    for i, f in enumerate(freqs):
        u = sdof(2 * np.pi * f, cfg['dt'], acc)
        z[i, :min(len(u), z.shape[1])] = np.abs(u[:z.shape[1]]) / (np.max(np.abs(u)) or 1.0)

    z -= z.min()
    z /= (z.max() or 1.0)
    return freqs, z

def save_diagram(freqs, z, out, cfg=CFG):
    plt.figure(figsize=cfg['size'], dpi=cfg['dpi'])
    plt.yscale('log')
    plt.ylim(freqs[0], freqs[-1])
    plt.xlim(0, cfg['duration'])
    plt.contourf(np.linspace(0, cfg['duration'], z.shape[1]), freqs,
                 z, levels=cfg['levels'], cmap='gray')
    plt.axis('off')
    plt.savefig(out, bbox_inches='tight', pad_inches=0)
    plt.close()


In [ ]:
from utils.utils import quantile_curve, label_group

def build_targets(midr, levels=(16, 84)):
    """Target response at the two exceedance levels."""
    return {p: quantile_curve(midr, p, method='empirical') for p in levels}

def batch(structs=None, limit=None):
    rows = []
    files = sorted(glob.glob(os.path.join(RAW_DIR, '*.txt')))
    for struct in (structs or TRAIN_SET):
        for f in (files[:limit] if limit else files):
            tag = f"{struct}_{os.path.splitext(os.path.basename(f))[0]}"
            try:
                freqs, z = response_diagram(f)
                save_diagram(freqs, z, os.path.join(RDTD_DIR, tag + '.png'))
                np.savetxt(os.path.join(RDTD_DIR, tag + '.txt'), z, fmt='%.4f')
                rows.append(dict(struct=struct, record=os.path.basename(f), tag=tag))
            except Exception as e:
                rows.append(dict(struct=struct, record=os.path.basename(f), error=str(e)))
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(TARGET_DIR, 'rdtd_index.csv'), index=False)
    return table

# midr: (n_records, n_floors) response of one group
# targets = build_targets(midr)
# labels, index = label_group(midr, targets[16])